In [16]:
"""
TMDB Trending Movies Scraper
Mengambil data dari endpoint Trending TMDB (max 1000 halaman = 20.000 film)
"""

import requests
import pandas as pd
import time

In [17]:
# ── CONFIG ────────────────────────────────────────────────────────────────
API_KEY  = "a7eaa982f861eda59593b7ebf834464c"
PAGES    = 1000
BASE_URL = "https://api.themoviedb.org/3"

In [18]:
def get_trending_ids(page):
    """Fetch satu halaman dari endpoint trending/movie/week."""
    r = requests.get(
        f"{BASE_URL}/trending/movie/week",
        params={"api_key": API_KEY, "language": "en-US", "page": page},
        timeout=10,
    )
    r.raise_for_status()
    return r.json().get("results", [])

In [19]:
def get_movie_details(movie_id):
    """Fetch detail lengkap + credits untuk satu film."""
    r = requests.get(
        f"{BASE_URL}/movie/{movie_id}",
        params={
            "api_key": API_KEY,
            "language": "en-US",
            "append_to_response": "credits,release_dates",
        },
        timeout=10,
    )
    r.raise_for_status()
    return r.json()

In [20]:
def parse_certificate(release_dates):
    """Ambil rating usia US (e.g. PG-13)."""
    for country in release_dates.get("results", []):
        if country.get("iso_3166_1") == "US":
            for rd in country.get("release_dates", []):
                cert = rd.get("certification", "").strip()
                if cert:
                    return cert
    return None

In [21]:
def parse_movie(data):
    """Ekstrak semua field dari response detail film."""
    title    = data.get("title")
    year     = (data.get("release_date") or "")[:4] or None
    overview = data.get("overview") or None
 
    certificate = parse_certificate(data.get("release_dates", {}))
 
    runtime  = data.get("runtime")
    duration = f"{runtime} min" if runtime else None
 
    genres = data.get("genres", [])
    genre  = ", ".join(g["name"] for g in genres)
 
    tmdb_rating = data.get("vote_average")
    votes       = data.get("vote_count")
 
    credits  = data.get("credits", {})
    crew     = credits.get("crew", [])
    cast     = credits.get("cast", [])
 
    director = next((p["name"] for p in crew if p.get("job") == "Director"), None)
    stars    = [p["name"] for p in cast[:4]]
 
    revenue = data.get("revenue")
    grossed = f"${revenue:,}" if revenue else None
 
    original_language = data.get("original_language")
    popularity        = data.get("popularity")
    tmdb_id           = data.get("id")
    imdb_id           = data.get("imdb_id")
 
    return {
        "Movie Name":        title,
        "Year":              year,
        "Certificate":       certificate,
        "Duration":          duration,
        "Genre":             genre,
        "TMDB Rating":       tmdb_rating,
        "Votes":             votes,
        "Director":          director,
        "Stars":             stars,
        "Grossed in $":      grossed,
        "Plot":              overview,
        "Original Language": original_language,
        "Popularity":        popularity,
        "TMDB ID":           tmdb_id,
        "IMDb ID":           imdb_id,
    }

In [ ]:
# ── Main ──────────────────────────────────────────────────────────────────
if __name__ == "__main__":

    # Step 1: kumpulkan semua movie ID dari trending
    all_ids = []
    for page in range(1, PAGES + 1):
        try:
            results = get_trending_ids(page)
            if not results:
                print(f"  Halaman {page}: tidak ada hasil, berhenti.")
                break
            all_ids.extend(m["id"] for m in results)
            if page % 100 == 0:
                print(f"  Halaman {page}/{PAGES} — {len(all_ids)} film terkumpul")
        except Exception as e:
            print(f"  Halaman {page} gagal: {e}")
        time.sleep(0.25)
 
    # Deduplikasi ID
    all_ids = list(dict.fromkeys(all_ids))
    print(f"\nTotal unique movie ID dari trending: {len(all_ids)}")
    print("Fetching detail tiap film...\n")
 
    # Step 2: fetch detail tiap film
    movie_data = []
    for idx, movie_id in enumerate(all_ids):
        try:
            data    = get_movie_details(movie_id)
            details = parse_movie(data)
            movie_data.append(details)
            if (idx + 1) % 500 == 0:
                print(f"[{idx+1}/{len(all_ids)}] ✓ {details['Movie Name']} ({details['Year']})")
        except Exception as e:
            print(f"[{idx+1}/{len(all_ids)}] ✗ ID {movie_id}: {e}")
        time.sleep(0.26)
    
    # Step 3: simpan ke parquet
    df = pd.DataFrame(movie_data)
    
    print(f"\nSelesai! Berhasil scraping {len(df)} film dari trending.")
    print(df[["Movie Name", "Year", "TMDB Rating", "Director"]].head(10).to_string())

    df.to_parquet("tmdb_movies_trending_raw.parquet")